# Taller 2 — Notebook 06: Interpretabilidad del modelo

**MINE-4101 · Predicción de churn para FinAlpes** · **Integrantes:** Daniel Triviño · Nicolás Bedoya

Entregable 4. Responde una pregunta que las métricas no responden: **¿por qué el modelo predice lo que predice?**
Sigue los pasos de la *Práctica 5: Interpretabilidad* del curso
([PracticasMachineLearning](https://datascience-uniandes.github.io/PracticasMachineLearning/talleres/practica-5/)),
que usa este mismo conjunto de datos. Se adapta al modelo elegido en el `04`, un XGBoost dentro de un pipeline, en
lugar del Random Forest de la práctica.

1. **Gráficos ICE** (comportamiento global y heterogeneidad entre clientes).
2. **LIME** (explicaciones locales de predicciones individuales, con su fidelidad y estabilidad).
3. **Valores SHAP** (explicaciones locales que, reunidas, dan una explicación global).
4. Referencia intrínsecamente interpretable (coeficientes de la regresión logística) y comparación de las técnicas.

> Ejecutar después de `04_modelado` (y preferiblemente del `05`).

In [ ]:
import json
import os

import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from tqdm import TqdmWarning
warnings.filterwarnings('ignore', category=TqdmWarning)   # aviso de barra de progreso al importar shap
import shap
from lime.lime_tabular import LimeTabularExplainer
from matplotlib.lines import Line2D
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.inspection import PartialDependenceDisplay, partial_dependence
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import StratifiedKFold, cross_val_predict

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
sns.set_theme(style='whitegrid')

RUTA_PROCESADOS = '../data/processed'
RUTA_MODELOS = f'{RUTA_PROCESADOS}/modelos'
RUTA_FIGS = '../figs'
os.makedirs(RUTA_FIGS, exist_ok=True)

SEMILLA = 42

## 1. Modelo, datos y función de predicción

Las técnicas de interpretabilidad cambian los valores de las variables y observan cómo cambia la predicción. Por eso
deben trabajar con las **variables originales** (edad en años, saldo en euros), no con las columnas ya imputadas,
escaladas y codificadas que recibe el XGBoost por dentro. Como el modelo del `04` es un pipeline que recibe las
variables originales, la función de predicción solo tiene que convertir lo que le llega en un `DataFrame` y devolver la
probabilidad de abandono (Práctica 5, Actividad 1, paso 1).

In [ ]:
df = pd.read_parquet(f'{RUTA_PROCESADOS}/churn_particion.parquet')
with open(f'{RUTA_MODELOS}/seleccion.json', encoding='utf-8') as f:
    seleccion = json.load(f)

OBJETIVO = 'churn'
VARIABLES_MODELO = seleccion['variables_modelo']
CATEGORICAS = ['products_number', 'credit_card', 'active_member']   # discretas o binarias
modelo = joblib.load(f'{RUTA_MODELOS}/xgboost.joblib')

X = df[VARIABLES_MODELO].astype(float)
y = df[OBJETIVO]
es_prueba = (df['particion'] == 'prueba').to_numpy()
X_train, y_train = X[~es_prueba], y[~es_prueba]
X_test, y_test = X[es_prueba], y[es_prueba]

def predecir_abandono(X):
    """Recibe las variables originales (DataFrame o arreglo de NumPy) y devuelve la probabilidad de abandono."""
    X = pd.DataFrame(np.asarray(X, dtype=float), columns=VARIABLES_MODELO)
    return modelo.predict_proba(X)[:, 1]

# Comprobación: la función predice exactamente lo mismo que el modelo
assert np.allclose(predecir_abandono(X_test.to_numpy()), modelo.predict_proba(X_test)[:, 1])
print(f'Modelo: {seleccion["modelo_elegido"]} {seleccion["hiperparametros"][seleccion["modelo_elegido"]]}')
print(f'Pasos del pipeline: {[nombre for nombre, _ in modelo.steps]}  |  la función reproduce sus predicciones en prueba')

**Muestras de trabajo.** LIME y los gráficos ICE no admiten valores faltantes en los registros que perturban. Por eso
las muestras se toman de clientes **sin faltantes**, que son \~89 % de cada conjunto. El modelo sí admite faltantes,
porque los imputa por dentro.

- **ICE:** 200 clientes de prueba.
- **SHAP:** 300 clientes de prueba, con un fondo de 100 clientes de entrenamiento.
- **Casos individuales:** salen de la muestra de SHAP, para poder explicar los mismos clientes con LIME y con SHAP.

El umbral de decisión es el mismo del `05`: el que maximiza el F1 en validación cruzada sobre entrenamiento. Se
recalcula aquí con las mismas particiones.

In [ ]:
X_test_completo = X_test.dropna()
X_train_completo = X_train.dropna()
print(f'Clientes sin faltantes: prueba {len(X_test_completo):,} de {len(X_test):,} | '
      f'entrenamiento {len(X_train_completo):,} de {len(X_train):,}')

X_muestra = X_test_completo.sample(300, random_state=SEMILLA)    # SHAP y casos individuales
X_ice = X_muestra.iloc[:200]                                       # ICE
X_fondo = X_train_completo.sample(100, random_state=SEMILLA)      # fondo de SHAP

# Umbral del 05: máximo F1 sobre predicciones de validación cruzada en entrenamiento
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
prob_cv = cross_val_predict(clone(modelo), X_train, y_train, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
precision, recall, umbrales = precision_recall_curve(y_train, prob_cv)
f1 = 2 * precision * recall / np.clip(precision + recall, 1e-12, None)
UMBRAL = umbrales[np.argmax(f1[:-1])]

prob_muestra = pd.Series(predecir_abandono(X_muestra), index=X_muestra.index, name='prob')
real_muestra = y.loc[X_muestra.index]
print(f'Umbral de decisión: {UMBRAL:.3f}  |  muestra de SHAP: {len(X_muestra)} clientes, '
      f'{real_muestra.sum()} abandonos ({real_muestra.mean() * 100:.1f}%)')

## 2. Gráficos ICE (Práctica 5, Actividad 1)

Una curva ICE muestra cómo cambia la probabilidad de abandono de **un** cliente si se varía una sola variable y se
dejan fijas las demás; hay una curva por cliente. Para usar `PartialDependenceDisplay` se envuelve la función de
predicción en un estimador mínimo de scikit-learn (código tomado de la ayuda *Gráfico ICE* del curso). La rejilla de
cada variable cubre del percentil 5 al 95; para `products_number` son sus valores posibles, de 1 a 4.

In [ ]:
class ClasificadorDesdeFuncion(ClassifierMixin, BaseEstimator):
    """Estimador mínimo que envuelve una función de predicción (ayuda "Gráfico ICE" del curso)."""
    def __init__(self, funcion=None):
        self.funcion = funcion
    def fit(self, X, y=None):
        self.classes_ = np.array([0, 1])
        return self
    def predict_proba(self, X):
        p = self.funcion(X)
        return np.column_stack([1 - p, p])
    def predict(self, X):
        return (self.funcion(X) >= 0.5).astype(int)

clasificador = ClasificadorDesdeFuncion(predecir_abandono).fit(X_train_completo)

def rejilla(col, n=30):
    """Valores de la rejilla: percentiles 5 a 95 (o los valores posibles en products_number)."""
    if col == 'products_number':
        return np.array([1.0, 2.0, 3.0, 4.0])
    return np.linspace(X_ice[col].quantile(0.05), X_ice[col].quantile(0.95), n)

COLS_ICE = ['age', 'balance', 'products_number']
VALORES = {X_ice.columns.get_loc(c): rejilla(c) for c in COLS_ICE}

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
PartialDependenceDisplay.from_estimator(clasificador, X_ice, COLS_ICE, kind='individual', custom_values=VALORES,
                                        ax=axes, ice_lines_kw={'alpha': 0.25, 'linewidth': 0.8})
fig.suptitle('Curvas ICE: probabilidad de abandono de 200 clientes de prueba', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{RUTA_FIGS}/06_ice.png', bbox_inches='tight'); plt.show()

Para leer las curvas con cifras se calcula, en cada punto de la rejilla, el **promedio** de las 200 curvas (la
dependencia parcial) y la dispersión entre clientes.

In [ ]:
def resumen_ice(col, puntos):
    pd_res = partial_dependence(clasificador, X_ice, [col], kind='individual', custom_values={col: rejilla(col)})
    curvas = pd.DataFrame(pd_res['individual'][0], columns=pd_res['grid_values'][0])
    sel = [curvas.columns[np.argmin(np.abs(curvas.columns - p))] for p in puntos]
    return pd.DataFrame({'valor': sel, 'prob_media': curvas[sel].mean().to_numpy(),
                         'p10': curvas[sel].quantile(0.10).to_numpy(), 'p90': curvas[sel].quantile(0.90).to_numpy()})

for col, puntos in [('age', [25, 35, 45, 55, 65]), ('balance', [0, 50_000, 100_000, 150_000]),
                    ('products_number', [1, 2, 3, 4])]:
    print(f'--- {col} ---'); display(resumen_ice(col, puntos).round(3))

**Qué vemos:**

- **Edad: el efecto más fuerte, y no lineal.** La probabilidad media pasa del 10 % a los 25 años y el 13 % a los 35, al
  32 % a los 45 y al 48 % hacia los 55. El efecto cambia mucho entre clientes: a los 55 años, el 10 % de los clientes
  con menos riesgo queda por debajo del 11 %, y el 10 % con más riesgo por encima del 84 %. Las curvas no son
  paralelas, así que el efecto de la edad depende de las demás variables.
- **Saldo: casi plano.** La probabilidad media se mueve entre 17 % y 21 % en todo el rango; el modelo casi no lo usa.
- **Número de productos: forma de U.** Tener 2 productos se asocia al menor riesgo (10 %), 1 producto lo sube (27 %), y
  3 o 4 lo disparan (73 % y 86 %). En los datos hay pocos clientes con 3 o 4 productos (266 y 60 de 9.985, según el
  `03`), así que llevar a un cliente cualquiera a 4 productos crea combinaciones poco vistas por el modelo.

### 2.1 ICE centrado

Las curvas parten de alturas distintas porque cada cliente tiene un riesgo distinto. El ICE centrado le resta a cada
curva su primer valor, así que todas parten de 0 y muestran solo **cuánto cambia** la probabilidad. Si las curvas
centradas se abren en abanico, el efecto de la edad no es el mismo para todos.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.8))
PartialDependenceDisplay.from_estimator(clasificador, X_ice, ['age'], kind='individual', centered=True,
                                        custom_values={X_ice.columns.get_loc('age'): rejilla('age')}, ax=ax,
                                        ice_lines_kw={'alpha': 0.25, 'linewidth': 0.8})
ax.set_title('ICE centrado de la edad (cambio de probabilidad desde la edad mínima de la rejilla)')
plt.tight_layout(); plt.savefig(f'{RUTA_FIGS}/06_ice_centrado_edad.png', bbox_inches='tight'); plt.show()

pd_c = partial_dependence(clasificador, X_ice, ['age'], kind='individual', custom_values={'age': rejilla('age')})
cambio = pd_c['individual'][0][:, -1] - pd_c['individual'][0][:, 0]
print(f'Cambio de probabilidad entre {rejilla("age")[0]:.0f} y {rejilla("age")[-1]:.0f} años: mediana {np.median(cambio):+.3f}, '
      f'percentil 10 {np.percentile(cambio, 10):+.3f}, percentil 90 {np.percentile(cambio, 90):+.3f}')

### 2.2 Curvas coloreadas por subgrupo: edad y miembro activo

Se repite el ICE de la edad coloreando las curvas según `active_member`, con la misma rejilla para los dos grupos
(código de la ayuda *Gráfico ICE*, sección "Colorear por subgrupo"). Si las formas difieren, las dos variables
interactúan.

In [ ]:
columna = 'age'
grupo = X_ice['active_member'] == 1
valores_edad = {X_ice.columns.get_loc(columna): rejilla(columna)}

fig, ax = plt.subplots(figsize=(7.5, 5))
display_ice = PartialDependenceDisplay.from_estimator(
    clasificador, X_ice[grupo], [columna], kind='individual', custom_values=valores_edad, ax=ax,
    ice_lines_kw={'color': 'tab:blue', 'alpha': 0.3, 'linewidth': 0.8})
PartialDependenceDisplay.from_estimator(
    clasificador, X_ice[~grupo], [columna], kind='individual', custom_values=valores_edad, ax=display_ice.axes_,
    ice_lines_kw={'color': 'tab:red', 'alpha': 0.3, 'linewidth': 0.8})
display_ice.axes_[0, 0].legend(handles=[Line2D([], [], color='tab:blue', label='miembro activo'),
                                        Line2D([], [], color='tab:red', label='miembro inactivo')])
display_ice.axes_[0, 0].set_title('ICE de la edad por actividad del cliente')
plt.tight_layout(); plt.savefig(f'{RUTA_FIGS}/06_ice_edad_actividad.png', bbox_inches='tight'); plt.show()

filas = []
for nombre, mascara in [('activo', grupo), ('inactivo', ~grupo)]:
    r = partial_dependence(clasificador, X_ice[mascara], ['age'], kind='average', custom_values={'age': rejilla('age')})
    curva = pd.Series(r['average'][0], index=r['grid_values'][0])
    filas.append({'grupo': nombre, 'clientes': int(mascara.sum()),
                  'prob_media_35': curva.iloc[np.argmin(np.abs(curva.index - 35))],
                  'prob_media_55': curva.iloc[np.argmin(np.abs(curva.index - 55))],
                  'max_prob_media': curva.max(), 'edad_del_max': curva.idxmax()})
display(pd.DataFrame(filas).set_index('grupo').round(3))

**Qué vemos:** sí hay **interacción entre edad y actividad**.

- **Activos:** la probabilidad media sube del 11 % a los 35 años al 30 % a los 55.
- **Inactivos:** sube del 15 % al 68 %, con un máximo del 74 % hacia los 57.

La edad pesa mucho más en los clientes inactivos. El ICE centrado muestra lo mismo: entre los 25 y los 58 años, la
mediana de los clientes gana 26 puntos de riesgo, pero el 10 % menos sensible gana solo 6 puntos y el 10 % más
sensible, 73. Esto explica parte de la ventaja del XGBoost sobre la regresión logística del `05`: un modelo lineal
asigna el mismo efecto de la edad a todos los clientes.

## 3. LIME (Práctica 5, Actividad 2)

LIME explica **una** predicción: genera miles de clientes parecidos al que se quiere explicar, los predice con el
modelo, pondera más a los más parecidos y ajusta un modelo lineal (el *sustituto*) que solo vale cerca de ese cliente.
Sus coeficientes son la explicación. Necesita la **función de predicción**, no el modelo, porque solo observa entradas
y salidas.

El explicador se crea con las variables originales de entrenamiento (sin faltantes) e indica cuáles son categóricas
(`products_number`, `credit_card`, `active_member`). Código de la ayuda *LIME* del curso, en modo clasificación.

In [ ]:
INDICES_CATEGORICAS = [VARIABLES_MODELO.index(c) for c in CATEGORICAS]

def predecir_dos_clases(X):
    p = predecir_abandono(X)
    return np.column_stack([1 - p, p])

explicador_lime = LimeTabularExplainer(X_train_completo.to_numpy(), feature_names=VARIABLES_MODELO,
                                       class_names=['se queda', 'abandona'], mode='classification',
                                       categorical_features=INDICES_CATEGORICAS, random_state=SEMILLA)

def explicar_lime(x, num_samples=5000, random_state=SEMILLA, num_features=6):
    explicador_lime.random_state = np.random.RandomState(random_state)
    return explicador_lime.explain_instance(x.to_numpy(), predecir_dos_clases, num_features=num_features,
                                            num_samples=num_samples)

### 3.1 Tres clientes: un acierto, un abandono no detectado y una falsa alarma

Con el umbral de decisión se clasifica a los 300 clientes de la muestra y se eligen tres casos:

- **Verdadero positivo (VP):** el cliente típico entre los que abandonan y el modelo detecta, es decir, el de
  probabilidad mediana dentro de ese grupo.
- **Falso negativo (FN):** el abandono que el modelo vio con **menos** riesgo; es el error más claro.
- **Falso positivo (FP):** el cliente que se queda y al que el modelo le dio **más** riesgo.

In [ ]:
pred_muestra = (prob_muestra >= UMBRAL).astype(int)
casos_vp = prob_muestra[(real_muestra == 1) & (pred_muestra == 1)]
casos_fn = prob_muestra[(real_muestra == 1) & (pred_muestra == 0)]
casos_fp = prob_muestra[(real_muestra == 0) & (pred_muestra == 1)]
CASOS = {
    'VP': (casos_vp - casos_vp.median()).abs().idxmin(),
    'FN': casos_fn.idxmin(),
    'FP': casos_fp.idxmax(),
}
print(f'En la muestra: VP={len(casos_vp)}, FN={len(casos_fn)}, FP={len(casos_fp)}')
tabla_casos = X_muestra.loc[list(CASOS.values())].copy()
tabla_casos.insert(0, 'caso', list(CASOS))
tabla_casos['prob_abandono'] = prob_muestra.loc[list(CASOS.values())].to_numpy()
tabla_casos['abandonó'] = real_muestra.loc[list(CASOS.values())].to_numpy()
display(tabla_casos.set_index('caso'))

In [ ]:
explicaciones_lime = {}
fig, axes = plt.subplots(1, 3, figsize=(18, 4.2))
for ax, (caso, idx) in zip(axes, CASOS.items()):
    exp = explicar_lime(X_muestra.loc[idx])
    explicaciones_lime[caso] = exp
    pesos = exp.as_list(label=1)[::-1]
    ax.barh([p[0] for p in pesos], [p[1] for p in pesos], color=['#C44E52' if p[1] > 0 else '#4C72B0' for p in pesos])
    ax.axvline(0, color='k', lw=0.8)
    ax.set_title(f'{caso}: prob. del modelo {prob_muestra.loc[idx]:.2f} | sustituto {exp.local_pred[0]:.2f}')
    ax.set_xlabel('peso en la probabilidad de abandono')
fig.suptitle('LIME: las 6 condiciones más influyentes (rojo sube el riesgo, azul lo baja)', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{RUTA_FIGS}/06_lime_casos.png', bbox_inches='tight'); plt.show()

for caso, exp in explicaciones_lime.items():
    print(f'--- {caso} ---')
    for condicion, peso in exp.as_list(label=1):
        print(f'  {condicion:35s} {peso:+.3f}')

**Qué vemos:**

- **VP** (50 años, inactivo, 1 producto, probabilidad 0,65). Lo empujan tener más de 44 años (+0,33), ser inactivo
  (+0,13) y tener un solo producto (+0,12). Es el perfil de riesgo que describen los gráficos ICE.
- **FN** (34 años, 2 productos, inactivo, probabilidad 0,04). Tener 2 productos (−0,17) y su edad (−0,13) pesan más
  que la inactividad (+0,11). Su perfil es el de los clientes que se quedan, y en estas 8 variables nada anticipaba
  su salida. Coincide con el `05`: los abandonos de clientes jóvenes con 2 productos son los que más se escapan.
- **FP** (51 años, inactivo, 1 producto, 10 años de antigüedad, probabilidad 0,68). Las mismas tres condiciones que en
  el VP y con pesos casi iguales; con estas variables es indistinguible de un cliente que abandona. Ese error no se
  corrige ajustando el modelo: requiere información que el conjunto no tiene, como transacciones, reclamos o uso de
  canales.

### 3.2 Fidelidad local

Si el sustituto lineal imita mal al modelo alrededor del cliente, la explicación no es confiable. Se compara la
predicción del modelo con la del sustituto (`local_pred`) y se revisa su R² local (`score`).

In [ ]:
fidelidad = pd.DataFrame([{'caso': caso, 'prob_modelo': prob_muestra.loc[CASOS[caso]],
                           'prob_sustituto': exp.local_pred[0], 'diferencia': exp.local_pred[0] - prob_muestra.loc[CASOS[caso]],
                           'R2_local': exp.score} for caso, exp in explicaciones_lime.items()]).set_index('caso')
display(fidelidad)

### 3.3 Estabilidad

Las perturbaciones son aleatorias, así que dos ejecuciones pueden dar explicaciones distintas. Se explica el caso VP
cinco veces con semillas distintas, primero con 1.000 perturbaciones y luego con 5.000, y se cuenta cuántas veces
aparece cada condición entre las tres más influyentes.

In [ ]:
filas = []
for num_samples in [1000, 5000]:
    for semilla in range(5):
        exp = explicar_lime(X_muestra.loc[CASOS['VP']], num_samples=num_samples, random_state=semilla)
        for rango, (condicion, peso) in enumerate(exp.as_list(label=1), start=1):
            filas.append({'num_samples': num_samples, 'semilla': semilla, 'rango': rango,
                          'condición': condicion, 'peso': peso})
estabilidad = pd.DataFrame(filas)
resumen_estab = (estabilidad.assign(en_top3=estabilidad['rango'] <= 3)
                 .groupby(['num_samples', 'condición'])
                 .agg(veces_en_top3=('en_top3', 'sum'), veces_en_top6=('rango', 'size'),
                      peso_medio=('peso', 'mean'), peso_desv=('peso', 'std'))
                 .sort_values(['num_samples', 'veces_en_top3', 'peso_medio'], ascending=[True, False, False]))
display(resumen_estab)

**Qué vemos:**

- **Fidelidad.** El sustituto se queda corto en el VP y el FP: predice 0,06 a 0,09 menos que el modelo, con un R² local
  de 0,55. En el FN imita peor (R² 0,29), así que esa explicación es la menos confiable. El sustituto es lineal y
  cerca de probabilidades muy bajas el modelo no lo es.
- **Estabilidad.** Las tres condiciones principales (edad > 44, inactivo, 1 producto) aparecen en el top 3 en las 10
  ejecuciones, con pesos que varían poco: desviación de hasta 0,012 con 1.000 perturbaciones y de hasta 0,005 con
  5.000. Las condiciones de peso pequeño cambian de orden e incluso de signo entre semillas; la antigüedad, por
  ejemplo, pasa de −0,012 a +0,001.

Conclusión: de una explicación de LIME solo conviene leer las condiciones principales, usando 5.000 perturbaciones.

### 3.4 Dos clientes con la misma probabilidad y perfiles distintos

¿El modelo llega a la misma probabilidad por las mismas razones? Se busca, entre los clientes de la muestra con riesgo
alto (probabilidad entre 0,4 y 0,7), el par con probabilidades casi iguales (diferencia menor a 0,02) y perfiles más
distintos (distancia entre sus variables estandarizadas).

In [ ]:
candidatos = prob_muestra[(prob_muestra > 0.4) & (prob_muestra < 0.7)].index
Z = (X_muestra.loc[candidatos] - X_train_completo.mean()) / X_train_completo.std()
mejor, mejor_dist = None, -1
for i, a in enumerate(candidatos):
    for b in candidatos[i + 1:]:
        if abs(prob_muestra[a] - prob_muestra[b]) < 0.02:
            dist = np.linalg.norm(Z.loc[a] - Z.loc[b])
            if dist > mejor_dist:
                mejor, mejor_dist = (a, b), dist
PAR = {'cliente A': mejor[0], 'cliente B': mejor[1]}
par = X_muestra.loc[list(PAR.values())].copy()
par.index = list(PAR)
par['prob_abandono'] = prob_muestra.loc[list(PAR.values())].to_numpy()
par['abandonó'] = real_muestra.loc[list(PAR.values())].to_numpy()
display(par)

for nombre, idx in PAR.items():
    exp = explicar_lime(X_muestra.loc[idx])
    print(f'--- {nombre} (prob. {prob_muestra[idx]:.3f}) ---')
    for condicion, peso in exp.as_list(label=1):
        print(f'  {condicion:35s} {peso:+.3f}')

**Qué vemos:** el cliente B es el VP de la sección 3.1; el A tiene 52 años, 2 productos, puntaje 452, 1 año de
antigüedad y es inactivo. Los dos tienen probabilidad 0,65, pero por caminos distintos:

- **Comparten** los aportes de la edad y la inactividad.
- **Se separan en los productos:** en B, tener un solo producto suma riesgo; en A, tener 2 productos le resta (−0,18).

LIME no muestra con claridad qué compensa esa resta en A. La sección 4 lo resuelve con SHAP, que reparte la
predicción exactamente. Además, A se quedó y B abandonó: la misma probabilidad no implica el mismo desenlace.

## 4. Valores SHAP (Práctica 5, Actividad 3)

Los valores SHAP reparten cada predicción entre las variables: cuánto aportó cada una a llevar la probabilidad desde un
**valor base** (la predicción promedio sobre la muestra de fondo) hasta la del cliente. Para cada cliente,
valor base + suma de sus valores SHAP = su predicción.

**Explicador.** La práctica usa el explicador de árboles con el Random Forest. Aquí se usa el **explicador agnóstico**
con la función de predicción y un fondo de 100 clientes de entrenamiento. Hay dos razones:

1. El XGBoost recibe columnas escaladas y codificadas, de modo que el explicador de árboles explicaría 11 columnas
   transformadas, no las 8 variables originales.
2. Con XGBoost, el explicador de árboles entrega log-odds, no probabilidades.

Con 8 variables, `shap` calcula los valores **exactos** (ExactExplainer), así que el costo es bajo. Los valores quedan
en **probabilidad de abandono**.

In [ ]:
explicador_shap = shap.Explainer(predecir_abandono, X_fondo)
valores = explicador_shap(X_muestra)
print(f'Explicador: {type(explicador_shap).__name__}  |  valores: {valores.values.shape}')

base = valores.base_values[0]
print(f'Valor base (predicción media sobre el fondo): {base:.3f}')
print(f'Predicción media del modelo en entrenamiento: {predecir_abandono(X_train).mean():.3f}  |  '
      f'tasa real de abandono en entrenamiento: {y_train.mean():.3f}')

# Aditividad: valor base + suma de los valores SHAP = predicción, para todos los clientes de la muestra
reconstruida = valores.base_values + valores.values.sum(axis=1)
print(f'Aditividad: diferencia máxima entre base + suma SHAP y la predicción = '
      f'{np.abs(reconstruida - prob_muestra.to_numpy()).max():.2e}')

**Qué vemos:** el valor base (0,191) es la predicción media sobre el fondo de 100 clientes de entrenamiento. Es
cercano, aunque no igual, a la tasa real de abandono (0,204) y a la predicción media del modelo en todo el
entrenamiento (0,204). La diferencia se debe a que el fondo es una muestra pequeña: otro fondo daría otro valor base. A
diferencia del Random Forest de la práctica, este modelo no corrigió el desbalance (`04`), así que no hay un
hiperparámetro que desplace el valor base. La aditividad se cumple exactamente (diferencia del orden de 10⁻¹⁵). Un
valor SHAP positivo sube la probabilidad de abandono respecto al promedio y uno negativo la baja; ambos están en
unidades de probabilidad.

### 4.1 Importancia global y diagrama de enjambre

- **Importancia global:** el promedio del valor SHAP absoluto de cada variable sobre los 300 clientes.
- **Diagrama de enjambre:** cada punto es un cliente. La posición horizontal es su valor SHAP; el color, el valor de la
  variable (rojo alto, azul bajo). Si los puntos rojos están a la derecha, los valores altos de esa variable suben el
  riesgo.

In [ ]:
importancia = pd.Series(np.abs(valores.values).mean(axis=0), index=VARIABLES_MODELO).sort_values(ascending=False)
display(importancia.rename('mean_abs_shap').to_frame().assign(pct_del_total=lambda t: t['mean_abs_shap'] / t['mean_abs_shap'].sum() * 100))

fig = plt.figure(figsize=(8, 5))
shap.plots.beeswarm(valores, max_display=8, show=False)
plt.title('SHAP: contribución de cada variable a la probabilidad de abandono', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{RUTA_FIGS}/06_shap_enjambre.png', bbox_inches='tight'); plt.show()

In [ ]:
# Dirección del efecto: valor SHAP medio por tramo de cada variable (lo que muestra el enjambre, con cifras)
def shap_por_tramo(col, bordes=None):
    v = pd.Series(valores[:, col].values, index=X_muestra.index)
    grupos = X_muestra[col] if bordes is None else pd.cut(X_muestra[col], bordes, right=False)
    return v.groupby(grupos, observed=True).agg(clientes='size', shap_medio='mean')

for col, bordes in [('products_number', None), ('active_member', None),
                    ('age', [18, 30, 40, 50, 60, 100]), ('balance', [-1, 1, 100_000, 300_000])]:
    print(f'--- {col} ---'); display(shap_por_tramo(col, bordes).round(3))

**Qué vemos:** el modelo descansa en **tres variables**.

| Variable | Peso en la importancia SHAP | Efecto |
|---|---|---|
| Edad | 38 % | Menores de 40: −0,09 a −0,11. De 50 a 59 años: +0,25. Mayores de 60: baja a +0,08. No es monótona, igual que en los ICE. |
| Número de productos | 33 % | 1 producto: +0,07. 2 productos: −0,09. 3 o 4: +0,49 y +0,70. |
| Actividad | 19 % | Inactivo: +0,06. Activo: −0,04. |

El saldo aporta el 6 % y es levemente positivo por encima de 100.000. El salario, el puntaje crediticio y la
antigüedad aportan menos del 3 % cada uno, y la tarjeta de crédito prácticamente nada. Coincide con los gráficos ICE,
también en la forma de U de `products_number`.

### 4.2 Interacción edad × actividad

El gráfico de dispersión de SHAP muestra el aporte de la edad de cada cliente, coloreado por `active_member`. Así se
contrasta con el ICE por subgrupo de la sección 2.2.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
shap.plots.scatter(valores[:, 'age'], color=valores[:, 'active_member'], ax=ax, show=False)
ax.set_title('SHAP de la edad, coloreado por miembro activo')
plt.tight_layout(); plt.savefig(f'{RUTA_FIGS}/06_shap_edad_actividad.png', bbox_inches='tight'); plt.show()

v_edad = pd.DataFrame({'edad': X_muestra['age'], 'shap_edad': valores[:, 'age'].values, 'activo': X_muestra['active_member']})
v_edad['tramo'] = pd.cut(v_edad['edad'], [18, 40, 50, 60, 100], right=False)
display(v_edad.groupby(['tramo', 'activo'], observed=True)['shap_edad'].agg(['size', 'mean']).unstack('activo').round(3))

**Qué vemos:** el aporte de la edad es mayor en los clientes inactivos dentro de cada tramo. De 50 a 59 años es +0,31 en
inactivos frente a +0,20 en activos; en mayores de 60, +0,17 frente a +0,06. Es la misma interacción que mostró el ICE
por subgrupo: dos técnicas distintas llegan a la misma conclusión.

### 4.3 Predicciones individuales: cascada y diagrama de fuerza

Los mismos tres clientes de la sección 3.1 (VP, FN, FP). La cascada parte del valor base, abajo, y suma una a una las
contribuciones de cada variable hasta llegar a la predicción del cliente, arriba. El diagrama de fuerza muestra lo
mismo en una sola línea.

In [ ]:
posicion = {caso: X_muestra.index.get_loc(idx) for caso, idx in CASOS.items()}
for caso, i in posicion.items():
    plt.figure()
    shap.plots.waterfall(valores[i], max_display=8, show=False)
    plt.title(f'{caso}: probabilidad {prob_muestra.iloc[i]:.2f} (abandonó: {int(real_muestra.iloc[i])})', fontweight='bold')
    plt.tight_layout(); plt.savefig(f'{RUTA_FIGS}/06_shap_cascada_{caso}.png', bbox_inches='tight'); plt.show()
    print(f'{caso}: base {valores.base_values[i]:.3f} + suma SHAP {valores.values[i].sum():+.3f} = '
          f'{valores.base_values[i] + valores.values[i].sum():.3f} (modelo: {prob_muestra.iloc[i]:.3f})')

In [ ]:
for caso, i in posicion.items():
    shap.plots.force(valores.base_values[i], valores.values[i], X_muestra.iloc[i].round(0), matplotlib=True,
                     show=False, text_rotation=15)
    plt.title(caso, loc='left', fontweight='bold', y=1.4)
    plt.show()

**SHAP frente a LIME en los tres casos.** Las tres variables con mayor aporte absoluto según cada técnica:

In [ ]:
filas = []
for caso, i in posicion.items():
    top_shap = pd.Series(valores.values[i], index=VARIABLES_MODELO).abs().sort_values(ascending=False).index[:3].tolist()
    condiciones_lime = [c for c, _ in explicaciones_lime[caso].as_list(label=1)]
    top_lime = []
    for condicion in condiciones_lime:
        var = next(v for v in sorted(VARIABLES_MODELO, key=len, reverse=True) if v in condicion)
        if var not in top_lime:
            top_lime.append(var)
    filas.append({'caso': caso, 'top3_SHAP': ', '.join(top_shap), 'top3_LIME': ', '.join(top_lime[:3]),
                  'coinciden': len(set(top_shap) & set(top_lime[:3]))})
display(pd.DataFrame(filas).set_index('caso'))

**Qué vemos:**

- En los tres casos, base + suma de SHAP reproduce la predicción.
- **VP:** +0,46 sobre la base.
- **FN:** −0,16, que lo deja en 0,04. La edad y los 2 productos lo hacen ver como un cliente que se queda.
- **FP:** +0,49, con las mismas variables que el VP.

**SHAP frente a LIME:** las tres variables principales coinciden en los tres casos; solo cambia el orden en el FN. Las
técnicas difieren en el tamaño de los aportes, no en las variables. LIME explica con intervalos (por ejemplo,
"edad > 44") y con un sustituto aproximado; SHAP, con el valor exacto del cliente y un reparto aditivo.

**El par de la sección 3.4 con SHAP.** Dos clientes con la misma probabilidad: SHAP reparte cada una exactamente.

In [ ]:
par_shap = pd.DataFrame({nombre: valores.values[X_muestra.index.get_loc(idx)] for nombre, idx in PAR.items()},
                        index=VARIABLES_MODELO)
par_shap = pd.concat([par_shap, pd.DataFrame({n: [base, base + par_shap[n].sum()] for n in PAR},
                                             index=['valor base', 'predicción'])])
display(par_shap.round(3))

**Qué vemos:** SHAP resuelve lo que LIME dejó abierto. En el cliente A, los 2 productos sí le restan riesgo, pero
menos de lo que decía LIME (−0,095 frente a −0,18). Lo compensan aportes que en B son menores o nulos:

- **Edad:** +0,35 en A frente a +0,25 en B, aunque solo los separan dos años. El efecto de la edad depende del resto
  del perfil.
- **Inactividad:** +0,15 frente a +0,11.
- **Saldo:** +0,05 frente a +0,00.

La misma probabilidad sale de combinaciones distintas.

### 4.4 Diagrama de fuerza conjunto

Gira los diagramas individuales de los 300 clientes y los pone uno al lado del otro, ordenados por similitud de sus
explicaciones. *Es interactivo: se ve al ejecutar el notebook en Jupyter o VS Code, no en la vista previa de GitHub.*

In [ ]:
shap.initjs()
shap.plots.force(valores.base_values[0], valores.values, X_muestra)

## 5. Referencia intrínsecamente interpretable: la regresión logística

La regresión logística del `04` es interpretable por construcción: cada coeficiente dice cuánto cambian las
*log-odds* de abandono por una desviación estándar de la variable (las numéricas están escaladas) o al pasar a esa
categoría. Su PR-AUC en prueba es menor (0,570 frente a 0,662, `05`). Sirve para contrastar si el XGBoost usa las
mismas variables, y qué capta además.

In [ ]:
logistica = joblib.load(f'{RUTA_MODELOS}/logistica.joblib')
preparar = logistica.named_steps['preparar']
categorias = preparar.named_transformers_['productos'].categories_[0]
nombres = (['credit_score', 'age', 'tenure', 'balance', 'estimated_salary']
           + [f'products_number = {int(c)}' for c in categorias] + ['credit_card', 'active_member'])
coefs = pd.DataFrame({'coeficiente': logistica.named_steps['modelo'].coef_[0]}, index=nombres)
coefs['odds_ratio'] = np.exp(coefs['coeficiente'])
display(coefs.sort_values('coeficiente', key=np.abs, ascending=False))

**Qué vemos:** la logística usa las mismas variables y en la misma dirección que el XGBoost. Las columnas de
`products_number` se leen solo en comparación entre sí, porque las cuatro categorías están en el modelo:

- **Productos:** 4 productos es la categoría de más riesgo, seguida de 3 y de 1; 2 productos es la de menor riesgo. Es
  la misma U.
- **Actividad:** ser activo multiplica las odds de abandono por 0,34.
- **Edad:** una desviación estándar más de edad las multiplica por 2,08.
- **Resto:** saldo, puntaje, antigüedad, tarjeta y salario tienen odds ratios cercanos a 1.

Lo que la logística **no** puede representar es lo que los gráficos ICE y SHAP muestran del XGBoost: que el riesgo baja
después de los 60 años y que la edad pesa más en los inactivos. Para captarlo, la logística necesitaría términos
cuadráticos e interacciones definidos a mano.

## 6. Comparación de las técnicas (Práctica 5, Actividad 3, paso 5)

| Técnica | Local o global | Pregunta que responde | Costo de cálculo | Estabilidad | Limitaciones |
|---|---|---|---|---|---|
| **ICE** | Local (una curva por cliente); su promedio es global | ¿Cómo cambiaría el riesgo de este cliente si cambiara solo esta variable? | Bajo: una predicción por cliente y punto de la rejilla | Determinista | Una variable a la vez; crea combinaciones irreales si la variable está correlacionada con otras; con muchas curvas se satura |
| **LIME** | Local | ¿Qué condiciones llevaron al modelo a esta predicción para este cliente? | Medio: miles de perturbaciones por cliente | Varía con la semilla y el número de muestras | Depende de cómo se perturba y se pondera; el sustituto puede imitar mal al modelo; explica intervalos, no valores |
| **SHAP** | Local, y global al reunir a muchos clientes | ¿Cuánto aportó cada variable a llevar el riesgo desde el promedio hasta el de este cliente? | Alto en general; bajo aquí (8 variables, cálculo exacto) | Determinista dado el fondo | El valor base depende del fondo; también evalúa combinaciones irreales; puede malinterpretarse como causal |

Ninguna de las tres demuestra **causalidad**. Describen cómo reacciona el **modelo** al cambiar sus entradas, no qué
pasaría con un cliente real si cambiara su edad o su número de productos. El modelo aprendió asociaciones de una
muestra observacional de clientes europeos.

## 7. Conclusiones

**Cómo funciona el modelo.** El XGBoost estima el riesgo de abandono casi solo con tres variables: edad, número de
productos y actividad, que suman el 90 % de la importancia SHAP.
- **Edad:** el riesgo sube con la edad hasta los 50–59 años y luego baja.
- **Productos:** el riesgo tiene forma de U. 2 productos es lo más protector; 1 producto sube el riesgo y 3 o 4 lo
  disparan.
- **Actividad:** la inactividad sube el riesgo y además amplifica el efecto de la edad. A los 55 años, el riesgo medio
  es del 68 % en inactivos y del 30 % en activos (ICE).
- **Resto:** el saldo pesa poco; el puntaje crediticio, el salario, la antigüedad y la tarjeta, casi nada.

Las tres técnicas (ICE, LIME y SHAP) y la regresión logística coinciden en las variables y en la dirección de los
efectos. Lo que el XGBoost capta y la logística no son las no linealidades y la interacción entre edad y actividad.

**Por qué se equivoca.** El abandono no detectado más claro es un cliente de 34 años con 2 productos, con el perfil de
quienes se quedan. La falsa alarma más clara es un cliente de 51 años, inactivo y con un producto, con el perfil de
quienes se van. Con estas 8 variables los dos casos son indistinguibles de los clientes "típicos" de su perfil. Para
reducir esos errores hacen falta datos de comportamiento (transacciones, reclamos, uso de canales), no otro algoritmo.

**Qué le diría un asesor a un cliente en riesgo.** Con la cascada de SHAP de ese cliente, el asesor puede decir, por
ejemplo: "su riesgo es alto sobre todo porque no usa activamente sus productos y tiene solo uno". De ahí salen dos
palancas sobre las que el banco puede actuar:
1. **Activar al cliente:** que use sus productos.
2. **Ofrecer un segundo producto.**

La edad sirve para priorizar a quién contactar, no es una palanca. Usarla para ofrecer un beneficio de retención es
distinto a usarla para negar un servicio, pero conviene auditarlo.

**Cuidados.**
- **No son causales.** Que el modelo asocie 2 productos con menos riesgo no prueba que vender un segundo producto
  retenga al cliente. El riesgo altísimo con 3 o 4 productos sugiere más bien que algo falla en esos clientes (por
  ejemplo, productos vendidos sin necesidad) y merece investigarse. Medir el efecto real de una acción exige un
  experimento, por ejemplo una campaña con grupo de control.
- **Las explicaciones tienen límites.** LIME es aproximado: su R² local va de 0,29 a 0,55 y solo sus condiciones
  principales son estables. SHAP depende del fondo elegido. Ambas evalúan combinaciones de variables que pueden no
  existir.
- **Qué mostrar a cada público.** Al banco, para explicar el riesgo de un cliente, la cascada de SHAP: es exacta,
  aditiva y está en probabilidad. A la gerencia, para el comportamiento global, los gráficos ICE y el diagrama de
  enjambre.